# LE04 – NumPy, Vektorisierung und Broadcasting
---
**Selbststudium** · Zeitbedarf ca. 4–5 Stunden · Phase B: Data Preparation

In LE03 haben Sie gesehen, dass pandas ganze Spalten auf einmal verrechnet, ohne Schleife. Das Fundament dafür ist [**NumPy**](https://numpy.org/) (*Numerical Python*). NumPy stellt schnelle, mehrdimensionale Arrays bereit und bildet die Basis fast aller Data-Science-Bibliotheken in Python: pandas, SciPy, Matplotlib, scikit-learn und viele mehr.

In dieser Einheit lernen Sie, **vektorisiert** zu denken: Statt Werte einzeln in einer Schleife zu bearbeiten, formulieren Sie Operationen für ganze Arrays und Spalten. Das macht Code kürzer, lesbarer und um ein Vielfaches schneller. Für Ihr Projekt ist das die Grundlage, um **abgeleitete Merkmale** zu berechnen: Dauern, Kategorien, Anteile, standardisierte Werte.

## Lernziele

Nach dieser Lehreinheit können Sie …

1. erklären, warum vektorisierte Operationen schneller sind als Python-Schleifen, und den Unterschied messen,
2. NumPy-Arrays erzeugen und ihre Eigenschaften `dtype`, `shape` und `ndim` bestimmen,
3. Arrays umformen und das Konzept der **Achsen** anwenden, insbesondere bei Aggregationen mit `axis`,
4. Arrays mit Indizes, Slices, Listen und booleschen Masken auswählen und den Unterschied zwischen **View** und **Kopie** erklären,
5. die **Broadcasting**-Regeln anwenden und vorhersagen, ob zwei Arrays kombinierbar sind,
6. Werte mit `np.where`, `np.select` und `np.clip` bedingt berechnen,
7. Schleifen, `iterrows` und `apply` in pandas durch vektorisierte Ausdrücke ersetzen.

## Inhalt

1. Warum NumPy? Eine Messung
2. Arrays erzeugen
3. Shape und Achsen
4. Auswählen: Indexing, Slicing, Masken
5. Views und Kopien
6. Rechnen mit Arrays
7. Broadcasting
8. Bedingte Werte
9. Fehlende Werte
10. Vektorisiert denken in pandas
11. Selbsttest
12. Zusammenfassung und Vorbereitung auf die Präsenz

## 1 Warum NumPy? Eine Messung
---

NumPy wird per Konvention als `np` importiert:

In [ ]:
import numpy as np
import pandas as pd

np.__version__

Angenommen, Sie haben eine Million Sessiondauern in Minuten und möchten sie in Stunden umrechnen. Mit einer Python-Liste brauchen Sie eine Schleife oder eine Comprehension, mit NumPy genügt ein Ausdruck für das ganze Array:

In [ ]:
rng = np.random.default_rng(42)
dauern_array = rng.integers(15, 240, size=1_000_000)   # 1 Mio. Zufallsdauern
dauern_liste = dauern_array.tolist()

In [ ]:
%timeit [d / 60 for d in dauern_liste]

In [ ]:
%timeit dauern_array / 60

Die vektorisierte Variante ist typischerweise **10- bis 100-mal schneller**. Die genauen Zahlen hängen von Ihrem Rechner ab. Die Gründe:

* **Einheitlicher Datentyp:** Alle Elemente eines Arrays haben denselben Typ (z. B. 64-Bit-Ganzzahlen) und liegen direkt hintereinander im Speicher. Eine Python-Liste enthält dagegen Verweise auf einzelne Objekte, deren Typ bei jedem Zugriff neu geprüft werden muss.
* **Kompilierte Schleifen:** Die Schleife über die Elemente läuft in optimiertem C-Code statt im Python-Interpreter.

Der Preis: Ein Array hat genau einen Datentyp und eine feste Größe. Für die typischen Aufgaben in Data Science, also viele gleichartige Werte, ist das genau richtig.

## 2 Arrays erzeugen
---

Das zentrale Objekt in NumPy ist das `ndarray` (*n-dimensionales Array*). Sie erzeugen es aus einer Liste, aus verschachtelten Listen für mehrere Dimensionen oder mit Hilfsfunktionen:

In [ ]:
a = np.array([95, 40, 105, 35])
print(a, a.dtype, type(a))

b = np.array([[1, 2, 3], [4, 5, 6]])     # zweidimensional: 2 Zeilen, 3 Spalten
print(b)

| Funktion | Ergebnis | Beispiel |
|---|---|---|
| `np.zeros(n)`, `np.ones(n)`, `np.full(n, wert)` | Array mit Nullen, Einsen bzw. einem festen Wert | `np.zeros((2, 3))` |
| `np.arange(start, stop, schritt)` | Zahlenfolge wie `range` | `np.arange(0, 60, 15)` → `[0 15 30 45]` |
| `np.linspace(start, stop, anzahl)` | gleichmäßig verteilte Werte, `stop` eingeschlossen | `np.linspace(0, 1, 5)` |
| `rng.integers`, `rng.normal`, `rng.choice` | Zufallszahlen | siehe unten |

In [ ]:
print(np.arange(0, 60, 15))
print(np.linspace(0, 1, 5))
print(np.zeros((2, 3)))

**Datentypen:** Weil alle Elemente denselben Typ haben, wandelt NumPy gemischte Eingaben um (*Upcasting*). Aus Ganzzahlen und einer Kommazahl wird ein `float64`-Array, und ein einziger String macht das ganze Array zu Text. Das ist derselbe Effekt wie beim `fokus`-Wert `"2,5"` in LE03:

In [ ]:
print(np.array([1, 2, 3.5]).dtype)
print(np.array([1, 2, "3"]))

**Zufallszahlen:** Zufallszahlen erzeugen Sie über einen *Generator*. Der Startwert (*Seed*) macht die Ergebnisse reproduzierbar. Das ist wichtig, wenn Sie Simulationen oder Stichproben in Ihrem Projekt verwenden.

In [ ]:
rng = np.random.default_rng(seed=2026)
print(rng.integers(1, 6, size=5))               # ganze Zahlen von 1 bis 5
print(rng.normal(loc=90, scale=20, size=3))     # normalverteilt, Mittelwert 90, Std. 20
print(rng.choice(["Mathe", "Statistik"], size=4))

💡 Ältere Beispiele verwenden `np.random.seed(…)` und `np.random.rand(…)`. Für neuen Code empfiehlt NumPy den Generator `np.random.default_rng`.

## 3 Shape und Achsen
---

Die **Form** (*shape*) eines Arrays gibt an, wie viele Elemente es entlang jeder Dimension hat. NumPy nennt die Dimensionen **Achsen** (*axes*).

Als durchgängiges Beispiel verwenden wir Kims Lernzeit pro Tag in den ersten vier Wochen. Vom 05.10. (Montag) bis zum 01.11.2026 (Sonntag) sind das genau 28 Tage. Die Aufbereitung mit pandas kennen Sie aus LE03. Tage ohne Session erhalten den Wert 0:

In [ ]:
df = pd.read_csv("data/lernsessions_2026-11-01.csv", parse_dates=["datum"])
tag = df["datum"].dt.strftime("%Y-%m-%d")
beginn = pd.to_datetime(tag + " " + df["start"])
ende = pd.to_datetime(tag + " " + df["ende"])
ende = ende.where(ende >= beginn, ende + pd.Timedelta(days=1))
df["dauer_min"] = ((ende - beginn).dt.total_seconds() / 60).astype(int)

pro_tag = (
    df.groupby("datum")["dauer_min"].sum()
    .reindex(pd.date_range("2026-10-05", "2026-11-01"), fill_value=0)
    .to_numpy(copy=True)                          # pandas -> NumPy (als veränderbare Kopie)
)
print(pro_tag.shape, pro_tag.dtype)
pro_tag

💡 Warum `copy=True`? In pandas 3 liefert `to_numpy()` oft ein **schreibgeschütztes** Array, das direkt auf die Daten des DataFrames zeigt. So schützt Copy-on-Write den DataFrame vor versehentlichen Änderungen. Zum Lesen und Rechnen genügt das. Wer die Werte im Array verändern will, fordert mit `copy=True` eine eigene Kopie an. Sonst erscheint die Fehlermeldung `assignment destination is read-only`.

Das ist ein eindimensionales Array mit 28 Werten. Mit `reshape` formen wir es in eine Matrix aus **4 Wochen × 7 Tagen** um. Die Anzahl der Elemente muss dabei gleich bleiben (4 · 7 = 28). Eine Dimension darf `-1` sein, NumPy berechnet sie dann selbst:

In [ ]:
lernzeit = pro_tag.reshape(4, 7)      # gleichwertig: pro_tag.reshape(-1, 7)
print(lernzeit.ndim, lernzeit.shape, lernzeit.size)
lernzeit

Jede Zeile ist eine Woche, jede Spalte ein Wochentag (Mo bis So). Die Zeilen bilden **Achse 0**, die Spalten **Achse 1**. Bei mehr Dimensionen kommen weitere Achsen hinzu. Ein Farbbild ist z. B. ein Array der Form `(Höhe, Breite, 3)`, das sehen Sie in der Präsenz.

<img src="img/achsen.png" width="650" alt="Achsen von NumPy-Arrays mit einer bis vier Dimensionen">

Die untere Zeile der Grafik zeigt die Achsen mit negativen Nummern: `-1` ist immer die letzte Achse, egal wie viele Dimensionen ein Array hat.

Weitere nützliche Umformungen:

* `lernzeit.T` vertauscht Zeilen und Spalten (Transponieren): 7 Tage × 4 Wochen.
* `lernzeit.ravel()` bzw. `flatten()` macht aus einem mehrdimensionalen wieder ein eindimensionales Array.
* `np.concatenate`, `np.vstack` und `np.hstack` fügen Arrays zusammen.

## 4 Auswählen: Indexing, Slicing, Masken
---

Bei mehreren Dimensionen trennt ein Komma die Achsen: `array[zeilen, spalten]`. Slices funktionieren wie bei Listen.

In [ ]:
print(lernzeit[0, 0])        # Woche 1, Montag
print(lernzeit[1])           # ganze Woche 2
print(lernzeit[:, 6])        # alle Sonntage
print(lernzeit[:, 5:])       # Wochenenden (Sa und So)
print(lernzeit[-1, ::2])     # letzte Woche, jeder zweite Tag

**Mit Listen auswählen** (*Fancy Indexing*): Eine Liste von Indizes wählt genau diese Elemente aus, auch in beliebiger Reihenfolge.

In [ ]:
lernzeit[:, [0, 2, 4]]       # Mo, Mi, Fr aller Wochen

**Boolesche Masken** kennen Sie aus pandas. Ein Vergleich liefert ein Array aus `True` und `False` mit derselben Form, und die Maske wählt die passenden Elemente aus. Mehrere Bedingungen werden wieder mit `&`, `|` und `~` verknüpft, jede in Klammern.

In [ ]:
maske = lernzeit > 240
print(maske)
print(lernzeit[maske])                         # Tage mit mehr als 4 Stunden
print(maske.sum(), "Tage mit mehr als 4 Stunden")
print(np.argwhere(lernzeit == 0))              # Positionen (Woche, Tag) der Tage ohne Lernen

💡 `maske.sum()` zählt die `True`-Werte, `maske.mean()` liefert ihren Anteil. Diesen Trick kennen Sie schon aus LE03.

## 5 Views und Kopien
---

Ein Slice eines NumPy-Arrays ist in der Regel eine **View**: kein neues Array, sondern ein Fenster auf dieselben Daten. Ändern Sie die View, ändert sich das Original mit:

In [ ]:
original = lernzeit[0, 0]        # Wert merken, um ihn später wiederherzustellen
woche1 = lernzeit[0]
woche1[0] = 999
print(lernzeit[0])               # das Original hat sich verändert!
print(pro_tag[:3])               # sogar pro_tag, denn auch reshape liefert eine View

In [ ]:
lernzeit[0, 0] = original        # Wert wiederherstellen
woche1 = lernzeit[0].copy()      # echte Kopie
woche1[0] = 999
print(lernzeit[0])               # unverändert

Merken Sie sich:

* **Slices** (`a[1:3]`, `a[:, 0]`) und in der Regel auch `reshape` liefern Views. Wenn Sie das Ergebnis verändern möchten, ohne das Original zu verändern, verwenden Sie `.copy()`.
* **Masken und Listen** (`a[a > 5]`, `a[[0, 2]]`) erzeugen immer eine Kopie.
* In **pandas 3** stellt sich die Frage nicht mehr: Wegen Copy-on-Write verhält sich jede Auswahl wie eine Kopie (LE03). Views spielen daher vor allem dann eine Rolle, wenn Sie direkt mit NumPy arbeiten.

## 6 Rechnen mit Arrays
---

**Elementweise Operationen:** Arithmetische Operatoren und Vergleiche wirken Element für Element. Zwei Arrays gleicher Form werden Position für Position verrechnet:

In [ ]:
geplant = np.array([120, 90, 120, 90, 60, 0, 60])     # Kims Wochenplan in Minuten
print(lernzeit[1] - geplant)                           # Abweichung in Woche 2
print(lernzeit[1] >= geplant)                          # Plan erfüllt?

**Universelle Funktionen** (*ufuncs*) wie `np.sqrt`, `np.abs`, `np.round`, `np.log` oder `np.exp` wenden eine mathematische Funktion auf jedes Element an:

In [ ]:
np.round(lernzeit / 60, 1)       # Stunden, auf eine Nachkommastelle

**Aggregationen** fassen viele Werte zu einem zusammen: `sum`, `mean`, `median`, `std`, `min`, `max`, `argmin`, `argmax`, `cumsum`. Ohne weitere Angabe beziehen sie sich auf das **ganze Array**. Mit `axis` legen Sie fest, **entlang welcher Achse** zusammengefasst wird. Diese Achse verschwindet im Ergebnis:

In [ ]:
print("gesamt:          ", lernzeit.sum())
print("je Wochentag:    ", lernzeit.sum(axis=0))    # über die Wochen (Achse 0) -> 7 Werte
print("je Woche:        ", lernzeit.sum(axis=1))    # über die Tage (Achse 1) -> 4 Werte
print("Ø je Wochentag:  ", lernzeit.mean(axis=0).round())

💡 **Eselsbrücke:** `axis=0` heißt „entlang der Zeilen zusammenfassen“, also **nach unten** rechnen. Das Ergebnis hat einen Wert je Spalte. `axis=1` rechnet **nach rechts** und liefert einen Wert je Zeile.

`argmax` liefert die **Position** des größten Werts. `cumsum` bildet die laufende Summe:

In [ ]:
wochentage = np.array(["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"])
print("Stärkster Tag je Woche:", wochentage[lernzeit.argmax(axis=1)])
print("Kumulierte Lernzeit (h) in Woche 1:", np.round(lernzeit[0].cumsum() / 60, 1))

⚠️ **Standardabweichung:** `np.std` teilt standardmäßig durch *n* (`ddof=0`), pandas mit `.std()` durch *n − 1* (`ddof=1`, Stichprobenstandardabweichung). Für dieselben Daten erhalten Sie also leicht unterschiedliche Werte. Mit `np.std(x, ddof=1)` rechnet NumPy wie pandas. Mehr dazu in LE08.

✏️ **Ausprobieren:** An welchem Wochentag hat Kim im Durchschnitt am meisten gelernt? In wie vielen Wochen hat Kim mehr als 20 Stunden gelernt?

In [ ]:
# Ihr Code

## 7 Broadcasting
---

Was passiert, wenn zwei Arrays **nicht** dieselbe Form haben? Im einfachsten Fall verrechnen Sie ein Array mit einer einzelnen Zahl, z. B. bei der Umrechnung von Meilen in Kilometer. NumPy „streckt“ die Zahl gedanklich auf die Form des Arrays:

<img src="img/broadcasting_skalar.png" width="600" alt="Ein Array wird mit der Zahl 1,6 multipliziert, die Zahl wird dafür auf die Form des Arrays gestreckt">

<small>Abbildung: [NumPy-Dokumentation](https://numpy.org/doc/stable/user/absolute_beginners.html), BSD-3-Clause-Lizenz</small>

Dieses Prinzip heißt **Broadcasting**. Es funktioniert nicht nur mit einzelnen Zahlen, sondern auch zwischen Arrays unterschiedlicher Form, solange sie **kompatibel** sind:

<img src="img/broadcasting.png" width="650" alt="Broadcasting eines 4x3-Arrays mit einem Array der Länge 3 und eines 4x1-Arrays mit einem Array der Länge 3">

### Die Regeln

NumPy vergleicht die Formen **von hinten** (von der letzten Achse aus). Zwei Achsen sind kompatibel, wenn

1. sie **gleich lang** sind oder
2. eine davon die **Länge 1** hat (sie wird gestreckt) oder
3. eine davon **fehlt** (sie wird vorne mit Länge 1 ergänzt).

| Form A | Form B | Ergebnis |
|---|---|---|
| `(4, 7)` | `()` (Zahl) | `(4, 7)` |
| `(4, 7)` | `(7,)` | `(4, 7)`: B wird für jede Zeile wiederholt |
| `(4, 7)` | `(4, 1)` | `(4, 7)`: B wird für jede Spalte wiederholt |
| `(4, 1)` | `(7,)` | `(4, 7)`: beide werden gestreckt |
| `(4, 7)` | `(4,)` | **Fehler:** 7 und 4 passen nicht |

### Beispiele mit Kims Lernzeit

**Abweichung vom Wochentagsdurchschnitt:** `lernzeit.mean(axis=0)` hat die Form `(7,)` und passt damit zur Form `(4, 7)`. Das Ergebnis zeigt für jeden Tag, wie viel Kim über oder unter dem üblichen Wert für diesen Wochentag lag:

In [ ]:
abweichung = lernzeit - lernzeit.mean(axis=0)
np.round(abweichung)

**Anteil jedes Tages an seiner Woche:** `lernzeit.sum(axis=1)` hat die Form `(4,)`. Das passt **nicht** zu `(4, 7)`, denn verglichen wird von hinten, also 4 mit 7:

In [ ]:
lernzeit / lernzeit.sum(axis=1)

Mit `keepdims=True` bleibt die zusammengefasste Achse mit Länge 1 erhalten. Die Form `(4, 1)` ist dann kompatibel:

In [ ]:
anteil = lernzeit / lernzeit.sum(axis=1, keepdims=True)
print(lernzeit.sum(axis=1, keepdims=True).shape)
np.round(anteil, 2)

💡 Alternativ fügen Sie mit `np.newaxis` (oder `None`) eine Achse der Länge 1 ein: `lernzeit.sum(axis=1)[:, np.newaxis]` hat ebenfalls die Form `(4, 1)`.

**Standardisieren (z-Werte):** Um Werte mit unterschiedlicher Größenordnung vergleichbar zu machen, zieht man den Mittelwert ab und teilt durch die Standardabweichung. Das Ergebnis gibt an, wie viele Standardabweichungen ein Wert vom Durchschnitt entfernt ist. Mit Broadcasting ist das eine Zeile, hier je Wochentag:

In [ ]:
z = (lernzeit - lernzeit.mean(axis=0)) / lernzeit.std(axis=0)
np.round(z, 1)

✏️ **Ausprobieren:** Kims Wochenplan `geplant` hat die Form `(7,)`. Berechnen Sie für **alle vier Wochen** auf einmal, an welchen Tagen Kim den Plan erfüllt hat. Wie oft insgesamt?

In [ ]:
# Ihr Code

## 8 Bedingte Werte
---

Statt `if`/`else` in einer Schleife verwenden Sie vektorisierte Funktionen:

* `np.where(bedingung, wert_wenn_wahr, wert_wenn_falsch)` wählt elementweise zwischen **zwei** Werten.
* `np.select([bedingung1, bedingung2, …], [wert1, wert2, …], default=…)` wählt zwischen **mehreren** Werten. Die erste zutreffende Bedingung gewinnt.
* `np.clip(a, min, max)` begrenzt Werte auf einen Bereich.

In [ ]:
dauern = df["dauer_min"].to_numpy()

lang_oder_nicht = np.where(dauern > 90, "lang", "normal")

kategorie = np.select(
    [dauern < 45, dauern <= 90],
    ["kurz", "mittel"],
    default="lang",
)
print(dauern[:6])
print(kategorie[:6])
print(np.clip(dauern, 30, 120)[:6])

Vergleichen Sie das mit LE01: Dort brauchten Sie für dieselbe Einteilung eine Schleife mit `if`, `elif` und `else`.

## 9 Fehlende Werte
---

NumPy kennzeichnet fehlende Kommazahlen mit `np.nan` (*Not a Number*). `NaN` ist „ansteckend“: Jede Rechnung mit `NaN` ergibt wieder `NaN`. Die `nan`-Varianten der Aggregationen ignorieren fehlende Werte:

In [ ]:
fokus = df["fokus_1bis5"].to_numpy()
print(fokus[:12])
print(np.mean(fokus), np.nanmean(fokus))
print(np.isnan(fokus).sum(), "fehlende Werte")

⚠️ `np.nan == np.nan` ergibt `False`. Prüfen Sie fehlende Werte deshalb immer mit `np.isnan` bzw. in pandas mit `isna()`. pandas-Methoden wie `mean()` überspringen fehlende Werte automatisch.

## 10 Vektorisiert denken in pandas
---

Jede numerische pandas-Spalte basiert auf einem NumPy-Array (`s.to_numpy()` liefert es). Deshalb gilt alles aus dieser Einheit auch für pandas: Rechnungen mit ganzen Spalten sind schnell, Schleifen über Zeilen sind langsam.

Drei Wege, für jede Session die Dauer in Stunden mit einem Aufschlag für Pausen zu berechnen, im Vergleich. Damit die Unterschiede messbar werden, vervielfachen wir Kims Daten auf 70 000 Zeilen:

In [ ]:
gross = pd.concat([df] * 1000, ignore_index=True)
len(gross)

In [ ]:
%%timeit -n 1 -r 3
# 1) Schleife mit iterrows
ergebnis = []
for _, zeile in gross.iterrows():
    ergebnis.append((zeile["dauer_min"] + 5 * zeile["pausen_anzahl"]) / 60)

In [ ]:
%%timeit -n 1 -r 3
# 2) apply mit einer Python-Funktion pro Zeile
ergebnis = gross.apply(lambda zeile: (zeile["dauer_min"] + 5 * zeile["pausen_anzahl"]) / 60, axis=1)

In [ ]:
%%timeit
# 3) vektorisiert
ergebnis = (gross["dauer_min"] + 5 * gross["pausen_anzahl"]) / 60

`apply` sieht elegant aus und ist etwas schneller als `iterrows`, ruft intern aber immer noch für jede Zeile eine Python-Funktion auf. Die vektorisierte Variante ist um **Größenordnungen** schneller, typischerweise etwa tausendmal. Bei den paar hundert Zeilen Ihres Projekts spielt die Laufzeit kaum eine Rolle. Vektorisierter Code ist aber auch **kürzer und lesbarer**, und er ist in der Praxis der Standard.

### Was statt einer Schleife?

| Ich möchte … | vektorisiert mit … |
|---|---|
| mit Spalten rechnen | `df["a"] * 60 + df["b"]` |
| Text bearbeiten | `.str` (LE03), z. B. `df["fach"].str.strip()` |
| Datum und Zeit zerlegen | `.dt` (LE03), z. B. `df["datum"].dt.dayofweek` |
| zwischen zwei Werten wählen | `np.where(bedingung, a, b)` oder `s.where(bedingung, anderer_wert)` |
| zwischen mehreren Werten wählen | `np.select(bedingungen, werte, default=…)` |
| Zahlen in Klassen einteilen | `pd.cut(s, bins=[…], labels=[…])` |
| Werte über eine Tabelle zuordnen | `s.map({"alt": "neu", …})` |
| je Gruppe rechnen | `groupby(…).transform(…)` (LE09) |

💡 **Faustregel:** Wenn Sie in pandas `for … in df.iterrows()` oder `apply(…, axis=1)` schreiben, gibt es fast immer eine vektorisierte Alternative. Suchen Sie danach, oder fragen Sie einen KI-Assistenten gezielt danach.

`pd.cut` teilt Zahlen in Klassen ein. `right=False` bedeutet, dass die rechte Grenze nicht mehr zur Klasse gehört:

In [ ]:
df["tageszeit"] = pd.cut(
    df["start"].str[:2].astype(int),
    bins=[0, 12, 18, 24],
    right=False,
    labels=["morgens", "nachmittags", "abends"],
)
df["wochenende"] = df["datum"].dt.dayofweek >= 5
df[["datum", "start", "tageszeit", "wochenende"]].head()

## 11 Selbsttest
---

In [ ]:
from dskurs.selbsttest import selbsttest

selbsttest("LE04")

## 12 Zusammenfassung und Vorbereitung auf die Präsenz
---

**Das Wichtigste in Kürze**

* NumPy-Arrays haben **einen** Datentyp und liegen kompakt im Speicher. Vektorisierte Operationen laufen in kompiliertem Code und sind dadurch viel schneller als Python-Schleifen.
* `shape` beschreibt die Form, Achse 0 sind die Zeilen, Achse 1 die Spalten. Aggregationen mit `axis=…` fassen entlang einer Achse zusammen, die dann im Ergebnis verschwindet.
* Auswahl mit `[zeilen, spalten]`, Listen und Masken. **Slices sind Views**, Masken und Listen erzeugen Kopien.
* **Broadcasting:** Formen werden von hinten verglichen. Passend sind gleiche Längen, die Länge 1 oder eine fehlende Achse. `keepdims=True` hilft beim zeilenweisen Rechnen.
* `np.where`, `np.select`, `np.clip` und `pd.cut` ersetzen `if`/`else` in Schleifen. `NaN` ist ansteckend, die `nan`-Funktionen und pandas ignorieren fehlende Werte.
* In pandas gilt: Spaltenoperationen statt `iterrows` oder `apply(…, axis=1)`.

**Checkliste bis zur Präsenz**

- [ ] Dieses Notebook läuft komplett durch (*Restart Kernel and Run All*).
- [ ] Den Selbsttest habe ich bearbeitet.
- [ ] Meine Datenerhebung läuft weiter.

### Weiterführende Quellen

* [NumPy: the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html), die offizielle Einführung
* [Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) in der NumPy-Dokumentation
* McKinney, W. (2022): [*Python for Data Analysis*](https://wesmckinney.com/book/), 3. Auflage, Kapitel 4 (NumPy Basics) und Anhang A (Advanced NumPy)
* VanderPlas, J. (2016): [*Python Data Science Handbook*](https://jakevdp.github.io/PythonDataScienceHandbook/), Kapitel 2 (online frei verfügbar)